# HW9: Supervised Learning with Spark MLlib

**ST554**
**David Pressley**

## Predicting Heart Disease with the UCI Cleveland Dataset

Use PySpark MLlib to fit and compare three classification models for predicting heart disease. The data comes from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/45/heart+disease) and includes 303 patient records from the Cleveland Clinic with 13 clinical features.

Predict whether a patient has heart disease (binary: present vs. absent) using three different model classes:

1. Logistic Regression
2. Decision Tree
3. Random Forest

Each model will be tuned with cross-validation, evaluated using AUC (Area Under the ROC Curve), and compared to select an overall best.

## Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("hw9").getOrCreate()

The UCI Cleveland dataset doesn't include column headers, so we assign them manually. Missing values are encoded as `?` in the raw file. The original target ranges from 0 (no disease) to 4, so we collapse it into a binary variable: 0 stays 0, and anything greater than 0 becomes 1.

In [ ]:
col_names = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'
]

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
pdf = pd.read_csv(url, names=col_names, na_values='?')

# a handful of rows have missing values in ca and thal
print(f"Shape before dropping NAs: {pdf.shape}")
pdf = pdf.dropna()
print(f"Shape after dropping NAs: {pdf.shape}")

# binarize target and rename to label
pdf['target'] = (pdf['target'] > 0).astype(int)
pdf = pdf.rename(columns={'target': 'label'})

# convert to Spark DataFrame
df = spark.createDataFrame(pdf)
df.show(5)

## Exploratory Data Analysis

In [ ]:
df.printSchema()

In [ ]:
df.describe().show()

In [ ]:
df.groupBy('label').count().show()

The classes are reasonably balanced. Here's a quick rundown of the 13 predictor features:

- **age**: age in years
- **sex**: 1 = male, 0 = female
- **cp**: chest pain type (0-3)
- **trestbps**: resting blood pressure (mm Hg)
- **chol**: serum cholesterol (mg/dl)
- **fbs**: fasting blood sugar > 120 mg/dl (1 = true, 0 = false)
- **restecg**: resting ECG results (0-2)
- **thalach**: maximum heart rate achieved
- **exang**: exercise-induced angina (1 = yes, 0 = no)
- **oldpeak**: ST depression induced by exercise
- **slope**: slope of the peak exercise ST segment (0-2)
- **ca**: number of major vessels colored by fluoroscopy (0-3)
- **thal**: thalassemia (3 = normal, 6 = fixed defect, 7 = reversible defect)